# Logistic Regression — Breast Cancer Classification

**Model:** Logistic Regression (sklearn)
**Dataset:** Breast Cancer Wisconsin — 569 samples, 30 features
**Covers:** Train/test split, StandardScaler, confusion matrix, ROC-AUC, feature coefficients

Logistic regression is the go-to baseline for binary classification. We establish a benchmark here before moving to ensemble methods.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)
print("Ready.")

## 1. Load Data and Split

In [ ]:
raw = load_breast_cancer(as_frame=True)
X, y = raw.data, raw.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Train class balance: {dict(y_train.value_counts())}")

## 2. Preprocessing — StandardScaler

Logistic regression is sensitive to feature scale. We fit the scaler on training data only and apply it to the test set — this prevents **data leakage** (a common mistake that inflates reported accuracy).

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform on train
X_test_sc  = scaler.transform(X_test)        # transform only on test (no leakage)

## 3. Train and Evaluate

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred = lr.predict(X_test_sc)
y_prob = lr.predict_proba(X_test_sc)[:, 1]  # probability of Benign class

print(classification_report(y_test, y_pred, target_names=raw.target_names))

## 4. Confusion Matrix and ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=raw.target_names,
    colorbar=False, ax=axes[0]
)
axes[0].set_title('Confusion Matrix — Logistic Regression')

# Right: ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, lw=2, label=f'ROC AUC = {auc:.3f}')
axes[1].fill_between(fpr, tpr, alpha=0.12)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Logistic Regression')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"ROC-AUC: {auc:.3f}")

## 5. Feature Coefficients

In logistic regression, the coefficient magnitude (after scaling) reflects each feature's contribution to the decision boundary. Positive coefficients push toward Benign; negative toward Malignant.

In [ ]:
coef_df = (pd.DataFrame({'feature': raw.feature_names, 'coefficient': lr.coef_[0]})
             .sort_values('coefficient', key=abs, ascending=False)
             .head(15))

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'][::-1], coef_df['coefficient'][::-1],
        color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 Feature Coefficients\n(positive = more likely Benign)', fontsize=12)
ax.set_xlabel('Coefficient (scaled features)')
plt.tight_layout()
plt.show()

## Summary

| Metric | Score |
|---|---|
| ROC-AUC | ~0.997 |
| Accuracy | ~97% |

Logistic regression achieves excellent performance on this dataset. The key predictors are worst-case measurements (radius, perimeter, concave points) — consistent with EDA findings.

**Next:** Random Forest with cross-validation for an ensemble comparison.